# XCoder-0.8B - Fine-tune di Colab T4 (v2: nama fail & folder tersuai)

**Panduan ringkas**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Sediakan data terkini (di PC):
   - `python datasets/scripts/merge_one.py --force` -> `datasets/all.jsonl`
   - Muat naik ke `MyDrive/DriveXcoder/xcoder/`: `all.jsonl` sahaja
     (`train_unsloth.py` diambil dari GitHub dalam sel 4 - sentiasa versi terkini;
      fallback ke salinan dalam folder staging jika muat turun gagal).
3. Edit **sel CONFIG** (sel pertama): nama data, folder output (lalai `.../xcoder/v2`),
   nama checkpoint, dan nama fail GGUF tersuai. Semua sel lain guna nilai itu.
4. Jalankan sel dari atas ke bawah. Training ~2.5-4.5 jam (1 epoch, packing, max_seq_length=3072).
5. Sesi putus? Buka semula dan run semula - auto-resume dari checkpoint terakhir di folder OUT_DIR.
   Penting: JANGAN jalankan dua sesi serentak ke folder yang sama (checkpoint boleh rosak).
6. Eksport tidak sempat (sesi mati selepas training)? Jalankan sel `--export-only` sahaja.
7. GGUF disalin dengan nama tersuai (GGUF_NAME) ke folder OUT_DIR - muat turun dari Drive.

**Beberapa akaun Google (sambung training antara akaun):**
1. Di Drive akaun A: kongsi folder `DriveXcoder/xcoder` -> tambah email akaun B sebagai Editor.
2. Di Drive akaun B ("Shared with me"): klik kanan folder itu -> "Add shortcut to Drive" ->
   letak dalam `MyDrive/DriveXcoder/` dengan nama `xcoder`.
3. Di Colab akaun B: buka notebook ini dan jalankan seperti biasa - laluan kekal,
   auto-resume dari checkpoint terakhir.


In [ ]:
# ===== CONFIG (edit di sini sahaja) =====
SRC = '/content/drive/MyDrive/DriveXcoder/xcoder'          # folder staging: data + train_unsloth.py
DATA_NAME = 'all.jsonl'                                    # nama fail data training (boleh tukar)
OUT_DIR = '/content/drive/MyDrive/DriveXcoder/xcoder/v2'   # folder output: checkpoint, eval, GGUF
OUT = 'xcoder-best-0.8b'                                   # nama checkpoint/merged dalam OUT_DIR
GGUF_NAME = 'xcoder-best-0.8b-q4_k_m.gguf'                 # nama fail GGUF tersuai (final)
# ========================================

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Salin data dari Drive, ambil train_unsloth.py dari GitHub (fallback: Drive), cipta OUT_DIR
import os, shutil, urllib.request

if not os.path.isdir(SRC):
    raise SystemExit(f'{SRC} tiada - buat folder DriveXcoder/xcoder (atau ubah SRC dalam sel CONFIG).')
os.makedirs(OUT_DIR, exist_ok=True)

SCRIPT_URL = ('https://raw.githubusercontent.com/qubexs/train-my-ai/master/'
              'finetune/train_unsloth.py')
try:
    urllib.request.urlretrieve(SCRIPT_URL, '/content/train_unsloth.py')
    print('train_unsloth.py: dari GitHub')
except Exception as e:
    src_script = f'{SRC}/train_unsloth.py'
    if not os.path.exists(src_script):
        raise SystemExit(f'muat turun GitHub gagal ({e}) dan tiada {src_script}')
    shutil.copy(src_script, '/content/train_unsloth.py')
    print('train_unsloth.py: dari Drive (fallback)')

if not os.path.exists(f'/content/{DATA_NAME}'):
    shutil.copy(f'{SRC}/{DATA_NAME}', f'/content/{DATA_NAME}')
print(DATA_NAME, round(os.path.getsize(f'/content/{DATA_NAME}') / 1e6, 1), 'MB')
print('OUT_DIR:', OUT_DIR, '| OUT:', OUT, '| GGUF:', GGUF_NAME)
os.chdir('/content')

In [ ]:
# Resipi rasmi Unsloth Qwen3.5 (pins tepat). Ambil masa ~5-10 min.
# Jika import unsloth gagal selepas ini: Runtime -> Restart session, teruskan dari sel seterusnya.
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"
print('install selesai')

In [ ]:
# Training penuh (checkpoint ke OUT_DIR setiap 100 langkah, auto-resume).
# CWD = /content -> merged/GGUF sementara ditulis ke /content/<OUT>-merged|-gguf,
# kemudian sel terakhir salin GGUF ke OUT_DIR dengan GGUF_NAME.
%cd /content
!python train_unsloth.py --data all --name {OUT} --out {OUT_DIR}

In [ ]:
# Jalankan SAHAJA jika training sudah selesai tetapi eksport merged/GGUF tidak sempat
%cd /content
!python train_unsloth.py --export-only --name {OUT} --out {OUT_DIR}

In [ ]:
# Salin GGUF dengan NAMA TERSUAI (GGUF_NAME) ke folder OUT_DIR
import glob, os, shutil

gs = glob.glob(f'/content/{OUT}-gguf/*.gguf')
if not gs:
    print('Tiada GGUF dijumpai - pastikan training + eksport selesai (sel sebelumnya)')
for f in gs:
    dst = os.path.join(OUT_DIR, GGUF_NAME)
    shutil.copy(f, dst)
    print('sedia:', dst, round(os.path.getsize(dst) / 1e6), 'MB')
print('Muat turun dari Drive:', OUT_DIR)

## Selepas ini (di PC)
1. Muat turun GGUF dari folder Drive (`OUT_DIR`) - nama ikut `GGUF_NAME`.
2. Daftar: `/model add <fail> --name xcoder-best --domains general`
3. Uji identiti: tanya "Siapa anda?" / "Are you Qwen?" -> "Saya XCoder..." tanpa sebut Qwen/Alibaba.
